# Stage 4A — Baseline Detector Inference

This notebook:
1. Loads `face_manifest.csv`
2. Runs a baseline deepfake detector on face crops
3. Aggregates frame-level scores to video-level scores
4. Saves:
   - `frame_detector_scores.csv`
   - `video_detector_scores.csv`

Use this notebook **before** the merge + experiments notebook.

## Assumption

This notebook assumes you already have a trained or pretrained **binary deepfake detector checkpoint**.

Typical label convention:
- `0` = real
- `1` = fake

You only need to adapt:
- `MODEL_NAME`
- `CHECKPOINT_PATH`
- checkpoint loading logic if your format differs

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoFeatureExtractor, AutoModelForImageClassification

/home/n.salikhova@innopolis.university/deepfake-emotion-robustness/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# =========================
# Configuration
# =========================
PROJECT_ROOT = Path(".").resolve().parent if Path(".").resolve().name == "notebooks" else Path(".").resolve()

SUBSET_NAME = "final"   # "pilot" or "final"

FACE_MANIFEST_PATH = PROJECT_ROOT / "datasets/metadata" / f"{SUBSET_NAME}_face_manifest.csv"
FRAME_SCORES_OUT   = PROJECT_ROOT / "datasets/detector_processed" / f"{SUBSET_NAME}_frame_detector_scores.csv"
VIDEO_SCORES_OUT   = PROJECT_ROOT / "datasets/detector_processed" / f"{SUBSET_NAME}_detector_scores.csv"

# ── Model ────────────────────────────────────────────────────────────────────
# Recommended: HuggingFace ViT-based deepfake detector (no registration needed)
HF_MODEL_ID = "dima806/deepfake_vs_real_image_detection"

# Academic standard alternative: Xception from FaceForensics++
# To use: set HF_MODEL_ID = None and fill in below
# MODEL_NAME    = "xception"
# CHECKPOINT_PATH = PROJECT_ROOT / "checkpoints" / "xception-hq.pth"

BATCH_SIZE  = 32
NUM_WORKERS = 2
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

# Aggregation
VIDEO_SCORE_MODE = "mean"   # "mean", "max", "topk_mean"
TOPK = 10

(PROJECT_ROOT / "datasets/detector_processed").mkdir(parents=True, exist_ok=True)

print("PROJECT_ROOT:", PROJECT_ROOT)
print("FACE_MANIFEST_PATH:", FACE_MANIFEST_PATH)
print("HF_MODEL_ID:", HF_MODEL_ID)
print("DEVICE:", DEVICE)


PROJECT_ROOT: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness
FACE_MANIFEST_PATH: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/metadata/final_face_manifest.csv
HF_MODEL_ID: dima806/deepfake_vs_real_image_detection
DEVICE: cuda


## Load face manifest

In [3]:
assert FACE_MANIFEST_PATH.exists(), f"Face manifest not found: {FACE_MANIFEST_PATH}"
face_df = pd.read_csv(FACE_MANIFEST_PATH)
print(f"Loaded {len(face_df):,} face rows from {FACE_MANIFEST_PATH}")
display(face_df.head())

Loaded 47,022 face rows from /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/metadata/final_face_manifest.csv


,video_id,frame_id,timestamp_sec,label,split,manipulation_family,manipulation_type,identity,source_subset,video_path,frame_path,face_id,face_path,bbox_x1,bbox_y1,bbox_x2,bbox_y2,det_score,face_width,face_height
0,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,0.0,fake,train,TalkingFace,EDTalk,id35,Celeb-synthesis,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,/home/n.salikhova@innopolis.university/deepfak...,36,13,209,235,0.731720,173,222
1,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,0.2,fake,train,TalkingFace,EDTalk,id35,Celeb-synthesis,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,/home/n.salikhova@innopolis.university/deepfak...,37,18,211,253,0.635672,174,235
2,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,0.4,fake,train,TalkingFace,EDTalk,id35,Celeb-synthesis,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,/home/n.salikhova@innopolis.university/deepfak...,44,7,215,241,0.737985,171,234
3,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,0.6,fake,train,TalkingFace,EDTalk,id35,Celeb-synthesis,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,/home/n.salikhova@innopolis.university/deepfak...,47,7,219,232,0.734100,172,225
4,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,0.8,fake,train,TalkingFace,EDTalk,id35,Celeb-synthesis,/home/n.salikhova@innopolis.university/deepfak...,/home/n.salikhova@innopolis.university/deepfak...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,/home/n.salikhova@innopolis.university/deepfak...,46,9,218,235,0.742079,172,226


## Dataset

In [4]:
class FaceDetectorDataset(Dataset):
    def __init__(self, df: pd.DataFrame, feature_extractor):
        self.df = df.reset_index(drop=True).copy()
        self.feature_extractor = feature_extractor

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        img = Image.open(row["face_path"]).convert("RGB")
        return {
            "image": img,
            "video_id":           str(row["video_id"]),
            "frame_id":           str(row["frame_id"]),
            "face_id":            str(row["face_id"]),
            "face_path":          str(row["face_path"]),
            "label":              str(row.get("label", "")),
            "split":              str(row.get("split", "")),
            "timestamp_sec":      float(row["timestamp_sec"]) if pd.notna(row.get("timestamp_sec")) else float("nan"),
            "manipulation_family": str(row.get("manipulation_family", "")),
            "manipulation_type":   str(row.get("manipulation_type", "")),
        }


def collate_fn(batch):
    """Custom collate: process images through feature_extractor as a batch."""
    return batch   # keep as list of dicts; we batch images inside inference


dataset = FaceDetectorDataset(face_df, feature_extractor=None)  # extractor set after model load
print("Dataset size:", len(dataset))


Dataset size: 47022


## Build model

This version uses `timm`.  
If your checkpoint was trained with another architecture, adapt this cell.

In [5]:
from transformers import AutoImageProcessor, AutoModelForImageClassification

processor = AutoImageProcessor.from_pretrained(HF_MODEL_ID)
model = AutoModelForImageClassification.from_pretrained(HF_MODEL_ID).to(DEVICE).eval()

# Determine which output index corresponds to "Fake"
fake_label_idx = next(
    (i for i, lbl in model.config.id2label.items() if lbl.strip().lower() == "fake"),
    1,  # fallback: index 1
)
print("Model labels :", model.config.id2label)
print("Fake label idx:", fake_label_idx)
print("Model loaded  :", HF_MODEL_ID)

# Attach processor to dataset and build DataLoader
dataset.feature_extractor = processor
loader = DataLoader(
    dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    collate_fn=lambda x: x,   # list-of-dicts; images batched inside inference fn
)
print(f"DataLoader ready — {len(loader)} batches")


Loading weights: 100%|██████████| 200/200 [00:00<00:00, 5680.88it/s]


Model labels : {0: 'Real', 1: 'Fake'}
Fake label idx: 1
Model loaded  : dima806/deepfake_vs_real_image_detection
DataLoader ready — 1470 batches


## Inference

In [6]:
@torch.no_grad()
def run_detector_inference(
    model: nn.Module,
    processor,
    loader: DataLoader,
    fake_idx: int = 1,
) -> pd.DataFrame:
    rows = []

    for batch in tqdm(loader, total=len(loader), desc="Running detector"):
        images = [item["image"] for item in batch]

        inputs = processor(images=images, return_tensors="pt").to(DEVICE)
        logits = model(**inputs).logits          # (B, num_classes)
        probs  = torch.softmax(logits, dim=-1)   # (B, num_classes)
        fake_probs = probs[:, fake_idx].cpu().numpy()

        for i, item in enumerate(batch):
            rows.append({
                "video_id":            item["video_id"],
                "frame_id":            item["frame_id"],
                "face_id":             item["face_id"],
                "face_path":           item["face_path"],
                "label":               item["label"],
                "split":               item["split"],
                "timestamp_sec":       item["timestamp_sec"],
                "manipulation_family": item["manipulation_family"],
                "manipulation_type":   item["manipulation_type"],
                "detector_score":      float(fake_probs[i]),
                "detector_pred":       int(fake_probs[i] >= 0.5),
            })

    return pd.DataFrame(rows)


frame_scores_df = run_detector_inference(model, processor, loader, fake_label_idx)
print(f"Generated {len(frame_scores_df):,} frame-level detector scores")
display(frame_scores_df.head())

frame_scores_df.to_csv(FRAME_SCORES_OUT, index=False)
print("Saved:", FRAME_SCORES_OUT)


Running detector: 100%|██████████| 1470/1470 [03:17<00:00,  7.45it/s]


Generated 47,022 frame-level detector scores


,video_id,frame_id,face_id,face_path,label,split,timestamp_sec,manipulation_family,manipulation_type,detector_score,detector_pred
0,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,/home/n.salikhova@innopolis.university/deepfak...,fake,train,0.0,TalkingFace,EDTalk,0.967139,1
1,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,/home/n.salikhova@innopolis.university/deepfak...,fake,train,0.2,TalkingFace,EDTalk,0.636166,1
2,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,/home/n.salikhova@innopolis.university/deepfak...,fake,train,0.4,TalkingFace,EDTalk,0.921823,1
3,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,/home/n.salikhova@innopolis.university/deepfak...,fake,train,0.6,TalkingFace,EDTalk,0.446197,0
4,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,Celeb-synthesis__TalkingFace__EDTalk__id35_000...,/home/n.salikhova@innopolis.university/deepfak...,fake,train,0.8,TalkingFace,EDTalk,0.772902,1


Saved: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/detector_processed/final_frame_detector_scores.csv


## Aggregate to video-level scores

In [7]:
def aggregate_video_scores(df: pd.DataFrame, mode: str = "mean", topk: int = 10) -> pd.DataFrame:
    rows = []

    for video_id, grp in tqdm(df.groupby("video_id"), desc="Aggregating video scores"):
        scores = grp["detector_score"].to_numpy()

        if mode == "mean":
            video_score = float(np.mean(scores))
        elif mode == "max":
            video_score = float(np.max(scores))
        elif mode == "topk_mean":
            k = min(topk, len(scores))
            top_scores = np.sort(scores)[-k:]
            video_score = float(np.mean(top_scores))
        else:
            raise ValueError(f"Unknown aggregation mode: {mode}")

        rows.append({
            "video_id": video_id,
            "label": grp["label"].iloc[0],
            "split": grp["split"].iloc[0],
            "manipulation_family": grp["manipulation_family"].iloc[0],
            "manipulation_type": grp["manipulation_type"].iloc[0],
            "n_face_frames": len(grp),
            "video_score_mode": mode,
            "detector_score": video_score,
            "detector_pred": int(video_score >= 0.5),
        })

    return pd.DataFrame(rows)

video_scores_df = aggregate_video_scores(frame_scores_df, mode=VIDEO_SCORE_MODE, topk=TOPK)
print(f"Generated {len(video_scores_df):,} video-level detector scores")
display(video_scores_df.head())

video_scores_df.to_csv(VIDEO_SCORES_OUT, index=False)
print("Saved:", VIDEO_SCORES_OUT)

Aggregating video scores: 100%|██████████| 790/790 [00:00<00:00, 4327.16it/s]

Generated 790 video-level detector scores


,video_id,label,split,manipulation_family,manipulation_type,n_face_frames,video_score_mode,detector_score,detector_pred
0,Celeb-real__id0_0002__9e73ac85,real,val,nan,Celeb-real,59,mean,0.930905,1
1,Celeb-real__id0_0005__c45f0938,real,val,nan,Celeb-real,77,mean,0.599537,1
2,Celeb-real__id0_0007__4f22f911,real,val,nan,Celeb-real,80,mean,0.374816,0
3,Celeb-real__id10_0000__9273e8bc,real,test,nan,Celeb-real,40,mean,0.841986,1
4,Celeb-real__id10_0002__6a16be4b,real,test,nan,Celeb-real,54,mean,0.349070,0


Saved: /home/n.salikhova@innopolis.university/deepfake-emotion-robustness/datasets/detector_processed/final_detector_scores.csv


## Quick checks

In [8]:
print("Frame-level rows:", len(frame_scores_df))
print("Video-level rows:", len(video_scores_df))
display(video_scores_df[["label", "split"]].value_counts().rename("count").reset_index())

Frame-level rows: 47022
Video-level rows: 790


,label,split,count
0,fake,train,292
1,real,train,278
2,real,test,79
3,fake,test,76
4,real,val,43
5,fake,val,22


## Next

After this notebook:
1. Merge
   - `manifest`
   - `video_emotion_features.csv`
   - `detector_scores.csv`
2. Run:
   - baseline metrics
   - emotion analysis
   - fusion model
   - ablation